# RoadScan v2 — YOLO detection experiment

This experiment starts from COCO-pretrained YOLO11s and uses exactly the 8,999 training images used by the ResNet-18 v2 baseline. Japan, India and Czech are training countries; China (1,977 images) is the validation domain. The United States (4,805 images) and local photographs are reserved for a separate final evaluation after the recipe and thresholds are fixed. No weights, predictions or processed labels from notebooks 04–13 are reused.

Detection mAP and image-level Macro-F1 answer different questions and are reported separately. Baseline reference: China Macro-F1 0.496, US Macro-F1 0.362. China results are development results, because checkpoint selection and threshold calibration use this set. No target score or absence of overfitting is guaranteed.

## 1. Validate and stage the fixed sample

The following cell checks image–label pairing, manifest equality, country counts, normalized YOLO boxes and exact image duplicates between training and validation. It copies data to local working storage for predictable training access. Labels remain unchanged. This audit does not exclude near-duplicate scenes; that remains a limitation.

In [ ]:
# Archived v2: disabled for the v3 background run; original results are in Version 1.
if False:
    from pathlib import Path
    from collections import Counter
    from concurrent.futures import ThreadPoolExecutor, as_completed
    import hashlib, json, shutil, time
    import numpy as np
    from PIL import Image
    import yaml
    
    BASE = Path('/kaggle/input/notebooks/sarvii7/roadscan-03-sample9000-v2')
    SOURCE = BASE / 'sample9000'
    ROOT = Path('/kaggle/working/sample9000')
    ART = Path('/kaggle/working/artifacts_v2')
    ART.mkdir(exist_ok=True)
    CLASSES = ['D00', 'D10', 'D20', 'D40']
    manifest = (BASE / 'sample9000_list.txt').read_text().splitlines()
    assert len(manifest) == len(set(manifest)) == 8999
    jobs = []
    for split, expected in [('train', 8999), ('val', 1977)]:
        images = {p.stem: p for p in (SOURCE/split/'images').glob('*.jpg')}
        labels = {p.stem: p for p in (SOURCE/split/'labels').glob('*.txt')}
        assert images.keys() == labels.keys() and len(images) == expected
        if split == 'train':
            assert set(images) == set(manifest)
            assert Counter(s.split('_')[0] for s in images) == {'Japan':4000, 'India':3500, 'Czech':1499}
        for folder in ['images', 'labels']:
            (ROOT/split/folder).mkdir(parents=True, exist_ok=True)
        jobs += [(split, s, images[s], labels[s]) for s in sorted(images)]
    
    def inspect_and_copy(job):
        split, stem, image, label = job
        raw = label.read_text()
        rows = [list(map(float, line.split())) for line in raw.splitlines() if line.strip()]
        for row in rows:
            assert len(row) == 5 and np.isfinite(row).all(), (split, stem, row)
            cls, x, y, w, h = row
            assert cls == int(cls) and 0 <= cls < 4
            assert 0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1
            assert x-w/2 >= -1e-5 and y-h/2 >= -1e-5 and x+w/2 <= 1.00001 and y+h/2 <= 1.00001
        data = image.read_bytes()
        target = ROOT/split/'images'/image.name
        target.write_bytes(data)
        with Image.open(target) as im:
            im.verify()
        (ROOT/split/'labels'/label.name).write_text(raw)
        return {'split':split, 'stem':stem, 'sha256':hashlib.sha256(data).hexdigest(), 'boxes':len(rows), 'classes':sorted({int(r[0]) for r in rows})}
    
    started = time.monotonic()
    records, errors = [], []
    with ThreadPoolExecutor(max_workers=24) as pool:
        futures = {pool.submit(inspect_and_copy,j):j[:2] for j in jobs}
        for f in as_completed(futures):
            try:
                records.append(f.result())
            except Exception as e:
                errors.append({'image':futures[f], 'error':repr(e)})
            if (len(records)+len(errors)) % 500 == 0:
                print('Audited:', len(records)+len(errors), '/', len(jobs), flush=True)
    (ART/'audit.json').write_text(json.dumps({'records':records, 'errors':errors}, indent=2))
    assert not errors, 'Audit failed: see artifacts_v2/audit.json'
    train_hash = {r['sha256'] for r in records if r['split']=='train'}
    val_hash = {r['sha256'] for r in records if r['split']=='val'}
    assert not train_hash & val_hash, 'Exact duplicate images cross train/validation. Resolve with team before training.'
    shutil.copy2(BASE/'sample9000_list.txt', ART/'sample9000_list.txt')
    DATA = Path('/kaggle/working/data_v2.yaml')
    DATA.write_text(yaml.safe_dump({'path':str(ROOT),'train':'train/images','val':'val/images','names':dict(enumerate(CLASSES))}, sort_keys=False))
    for split in ['train','val']:
        subset = [r for r in records if r['split']==split]
        print(split, 'images:',len(subset), 'boxes:',sum(r['boxes'] for r in subset), 'empty:',sum(r['boxes']==0 for r in subset))
    print('Audit passed. Minutes:', round((time.monotonic()-started)/60,1), flush=True)
    

## 2. Fine-tune a COCO-pretrained detector

YOLO11s at 640 pixels provides a practical first run before testing larger resolutions. All trainable layers are fine-tuned using AdamW, cosine learning-rate decay, three warmup epochs and weight decay. Mosaic, horizontal flips, scale and HSV changes are applied to training images only. Small rotations are avoided because D00 and D10 depend on crack direction.

The run allows 40 epochs with early stopping after 8 epochs without validation improvement. A callback requests a stop at the next completed epoch once 90 minutes have elapsed; validation and artifact export add time. Epoch scheduling remains fixed. The checkpoint with the best validation detection fitness is selected. GPU availability is required. PyTorch is reused from Kaggle rather than reinstalled.

References: https://docs.ultralytics.com/models/yolo11/ and https://docs.ultralytics.com/modes/train/

In [ ]:
# Archived v2: disabled for the v3 background run; original results are in Version 1.
if False:
    import sys, subprocess, importlib.metadata, platform
    try:
        importlib.metadata.version('ultralytics')
    except importlib.metadata.PackageNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.177'])
    import torch, ultralytics
    from ultralytics import YOLO
    assert DATA.exists() and len(records) == 10976 and not errors
    assert torch.cuda.is_available(), 'Select a GPU accelerator before running training.'
    versions = {'python':platform.python_version(), 'torch':torch.__version__, 'ultralytics':ultralytics.__version__, 'gpu':torch.cuda.get_device_name(0)}
    (ART/'environment.json').write_text(json.dumps(versions, indent=2))
    print(versions)
    model = YOLO('yolo11s.pt')
    clock = {}
    def start_clock(trainer):
        clock['start'] = time.monotonic()
    def budget_stop(trainer):
        if time.monotonic() - clock['start'] >= 90*60:
            trainer.stop = True
            print('90-minute training budget reached; finishing this epoch and saving.')
    model.add_callback('on_train_start', start_clock)
    model.add_callback('on_fit_epoch_end', budget_stop)
    train_results = model.train(
        data=str(DATA), imgsz=640, epochs=40, patience=8, batch=16,
        device=0, workers=2, seed=42, deterministic=True,
        optimizer='AdamW', lr0=0.001, lrf=0.01, cos_lr=True,
        warmup_epochs=3, weight_decay=0.0005,
        mosaic=1.0, close_mosaic=10, mixup=0.0,
        scale=0.5, translate=0.1, degrees=0.0, fliplr=0.5, flipud=0.0,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        cache=False, amp=True, save=True, save_period=5, plots=True,
        project='/kaggle/working/runs_v2', name='yolo11s_640_seed42',
    )
    RUN = Path(model.trainer.save_dir)
    BEST = RUN/'weights/best.pt'
    assert BEST.exists()
    shutil.copy2(BEST, ART/'yolo_v2_best.pt')
    shutil.copy2(RUN/'results.csv', ART/'training_history.csv')
    shutil.copy2(RUN/'args.yaml', ART/'training_args.yaml')
    print('Best model saved:', ART/'yolo_v2_best.pt')
    

## 3. Evaluate detection and calibrate image-level decisions

Detection is evaluated on the complete China validation set using standard mAP50 and mAP50–95. For the ResNet comparison, each image receives four scores: the maximum retained detection confidence for each class, or zero when no detection is present. Predictions are retained down to confidence 0.001; this inference floor is fixed before evaluation.

Class thresholds are selected on China from 0.05 to 0.95 in steps of 0.05, matching the baseline grid. Both fixed-threshold (0.5) and calibrated Macro-F1 are saved, together with AUROC, per-class precision/recall/F1, and the always-positive reference. Calibrated validation F1 is optimistic and is not an independent test result. The saved thresholds must be reused unchanged on the United States and local photographs. Training history and country-specific training diagnostics will inform follow-up work; this run alone cannot establish generalization to every city.

In [ ]:
# Archived v2: disabled for the v3 background run; original results are in Version 1.
if False:
    import pandas as pd
    from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
    best_model = YOLO(str(ART/'yolo_v2_best.pt'))
    detection = best_model.val(data=str(DATA), split='val', imgsz=640, batch=16, device=0, workers=2, plots=True, project='/kaggle/working/evaluation_v2', name='china')
    (ART/'china_detection.json').write_text(json.dumps({str(k):float(v) for k,v in detection.results_dict.items()}, indent=2))
    stems = sorted(p.stem for p in (ROOT/'val/labels').glob('*.txt'))
    truth = np.zeros((len(stems),4), dtype=int)
    scores = np.zeros((len(stems),4), dtype=float)
    lookup = {stem:i for i,stem in enumerate(stems)}
    for i,stem in enumerate(stems):
        for line in (ROOT/'val/labels'/f'{stem}.txt').read_text().splitlines():
            if line.strip():
                truth[i,int(line.split()[0])] = 1
    for result in best_model.predict(source=str(ROOT/'val/images'), imgsz=640, conf=0.001, iou=0.7, max_det=300, device=0, stream=True, verbose=False):
        i = lookup[Path(result.path).stem]
        for cls, conf in zip(result.boxes.cls.cpu().numpy().astype(int), result.boxes.conf.cpu().numpy()):
            scores[i,cls] = max(scores[i,cls], float(conf))
    grid = np.arange(0.05,0.96,0.05)
    thresholds = []
    for c in range(4):
        values = [precision_recall_fscore_support(truth[:,c], scores[:,c]>=t, average='binary', zero_division=0)[2] for t in grid]
        thresholds.append(float(grid[int(np.argmax(values))]))
    report = []
    for mode, threshold in [('fixed_0.5', np.full(4,0.5)),('calibrated',np.array(thresholds))]:
        p,r,f,_ = precision_recall_fscore_support(truth,scores>=threshold,zero_division=0)
        for c,name in enumerate(CLASSES):
            prevalence = truth[:,c].mean()
            auc = roc_auc_score(truth[:,c],scores[:,c]) if len(np.unique(truth[:,c]))==2 else None
            report.append({'domain':'China','mode':mode,'class':name,'threshold':float(threshold[c]),'precision':float(p[c]),'recall':float(r[c]),'f1':float(f[c]),'auroc':auc,'always_positive_f1':float(2*prevalence/(1+prevalence))})
        print(mode,'Macro-F1:',round(float(f.mean()),4))
    pd.DataFrame(report).to_csv(ART/'china_image_metrics.csv',index=False)
    np.savez_compressed(ART/'china_image_scores.npz',stems=np.array(stems),truth=truth,scores=scores)
    (ART/'inference_config.json').write_text(json.dumps({'classes':CLASSES,'thresholds':thresholds,'imgsz':640,'conf_floor':0.001,'iou':0.7,'max_det':300,'seed':42,'threshold_source':'China validation only'},indent=2))
    print(pd.DataFrame(report).to_string(index=False))
    print('Saved artifacts:',sorted(p.name for p in ART.iterdir()))
    print('Next: freeze recipe and evaluate the US and local test images. No test scores were used here.')
    

## 4. Expanded-data experiment — CPU preparation (v3)

Independent section in the **same notebook**. The original v2 experiment and saved results remain available. Run only this section on CPU for data preparation; do not use Run All, which also includes the earlier v2 training cells.

This candidate uses all Japan, India and Czech training images plus part of China MotorBike. Split whole groups of perceptually similar images, with separate validation for checkpoint selection and calibration for thresholds. US/local images stay outside development. ResNet must be rerun on the exact exported manifests for a fair comparison.

**Preparation completed:** 22,423 train / 298 validation / 297 calibration images. No detected exact/perceptual match crosses the splits. Visual similarity is a heuristic, so route/scene independence still needs review before training. No GPU training has started.

### 4.1 Inspect the full source dataset


In [ ]:
from pathlib import Path
import json
V3_OUT = Path('/kaggle/working/roadscan_v3_split')
V3_OUT.mkdir(exist_ok=True)
V3_SOURCE = Path('/kaggle/input/datasets/sarvii7/roadscan-yolo/RoadScan_yolo')
assert V3_SOURCE.is_dir(), str(V3_SOURCE)
print('Full source:', V3_SOURCE, flush=True)
for folder in sorted(V3_SOURCE.iterdir()):
    if folder.is_dir():
        print(folder.name, [p.name for p in folder.iterdir()][:8], flush=True)
print('Source inspection complete.', flush=True)


### 4.2 Build and audit the provisional visual-group split

All **23,018 development images** passed decoding, image-label pairing and bounding-box validation.

| Split | Japan | India | Czech | China MotorBike | Total |
|---|---:|---:|---:|---:|---:|
| Train | 10,506 | 7,706 | 2,829 | 1,382 | **22,423** |
| Validation (checkpoint selection) | 0 | 0 | 0 | 298 | **298** |
| Calibration (thresholds only) | 0 | 0 | 0 | 297 | **297** |

The final candidate groups exact image duplicates and 64-bit perceptual-hash matches (Hamming distance <= 6) into connected components before splitting with seed 42. China has 1,473 visual groups; 1,859 matching pairs were detected across the development data. All detected pairs stay in one split. Four classes are present in every split, including D40 (16 validation images, 29 calibration images).

Arbitrary 50-image filename blocks overconnected China and were discarded. The final candidate uses visual groups; it does not claim verified route/sequence independence. A sampled contact sheet was visually inspected and showed strongly similar road patches. This cannot rule out all related scenes or missed duplicates. The split remains provisional pending broader scene review; no GPU training has started.

Artifacts: manifest.csv plus train/val/calibration_manifest.csv (same assignments for YOLO and ResNet), data_v3.yaml and image lists, class_distribution.csv, audit.json, near_duplicate_pairs.csv and similarity_review.jpg. Source-relative manifests are portable; regenerate absolute lists if the Kaggle input mount changes. Re-run the baseline on these exact assignments. US/local images were not opened.


In [ ]:
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
import hashlib, re, random, csv, time
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageDraw
from scipy.fft import dctn
import yaml

V3_CLASSES = ['D00', 'D10', 'D20', 'D40']
V3_COUNTRIES = ['Japan', 'India', 'Czech', 'China_MotorBike']
V3_SEED, V3_BLOCK, V3_RADIUS = 42, 50, 6
v3_start = time.monotonic()
def country_of(stem):
    return next((c for c in V3_COUNTRIES if stem.startswith(c + '_')), None)
def audit_image(p):
    country = country_of(p.stem)
    label = p.parent.parent/'labels'/(p.stem + '.txt')
    assert label.exists(), f'Missing label: {p}'
    rows = [list(map(float, s.split())) for s in label.read_text().splitlines() if s.strip()]
    present = np.zeros(4, dtype=int)
    for row in rows:
        assert len(row) == 5 and np.isfinite(row).all(), str(label)
        c,x,y,w,h = row
        assert c == int(c) and 0 <= c < 4 and 0 < w <= 1 and 0 < h <= 1
        assert x-w/2 >= -1e-4 and y-h/2 >= -1e-4 and x+w/2 <= 1.0001 and y+h/2 <= 1.0001, str(label)
        present[int(c)] = 1
    with Image.open(p) as im:
        im.load()
        a = np.asarray(im.convert('L').resize((32,32)), dtype=float)
        low = dctn(a, norm='ortho')[:8,:8].flatten()
        bits = low > np.median(low[1:])
        phash = int.from_bytes(np.packbits(bits).tobytes(), 'big')
    return dict(image=p, label=label, stem=p.stem, country=country, exact=hashlib.sha256(p.read_bytes()).hexdigest(), phash=phash, boxes=len(rows), present=present)
paths = sorted(p for split in ['train','val'] for p in (V3_SOURCE/split/'images').glob('*.jpg') if country_of(p.stem))
assert paths and len({p.stem for p in paths}) == len(paths), 'Duplicate stems or no development images'
print('Auditing development images:', len(paths), flush=True)
if 'v3_records' in globals() and [r['image'] for r in v3_records] == paths:
    print('Reusing the completed audit from this CPU session.',flush=True)
else:
    with ThreadPoolExecutor(max_workers=4) as pool:
        v3_records = list(pool.map(audit_image, paths))
print('Audit passed:', dict(Counter(r['country'] for r in v3_records)), flush=True)
parent = list(range(len(v3_records)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
def union(a,b):
    a,b = find(a),find(b)
    if a != b: parent[max(a,b)] = min(a,b)
exact_seen, blocks = {}, {}
for i,r in enumerate(v3_records):
    if r['exact'] in exact_seen: union(i, exact_seen[r['exact']])
    else: exact_seen[r['exact']] = i
# Seven disjoint hash chunks guarantee candidate coverage for Hamming radius <= 6.
# Any two 64-bit hashes at distance <= 6 share at least one of these chunks.
assert V3_RADIUS == 6
buckets = [defaultdict(list) for _ in range(7)]
representatives = {}
near_pairs = []
chunk_sizes = [9,9,9,9,9,9,10]
for i,r in enumerate(v3_records):
    value = r['phash']
    if value in representatives:
        j = representatives[value]
        union(i,j)
        near_pairs.append((i,j,0))
        continue
    keys, offset = [], 0
    candidates = set()
    for k,size in enumerate(chunk_sizes):
        key = (value >> offset) & ((1 << size)-1)
        keys.append(key)
        candidates.update(buckets[k].get(key, []))
        offset += size
    for j in sorted(candidates):
        distance = (value ^ v3_records[j]['phash']).bit_count()
        if distance <= V3_RADIUS:
            union(i,j)
            near_pairs.append((i,j,distance))
    representatives[value] = i
    for k,key in enumerate(keys): buckets[k][key].append(i)
    if i % 5000 == 0: print('Similarity search:',i,'/',len(v3_records),flush=True)
groups = defaultdict(list)
for i in range(len(v3_records)): groups[find(i)].append(i)
forced = {g for g,ids in groups.items() if any(v3_records[i]['country'] != 'China_MotorBike' for i in ids)}
china_groups = sorted(set(groups)-forced)
random.Random(V3_SEED).shuffle(china_groups)
china_total = sum(r['country']=='China_MotorBike' for r in v3_records)
print('China visual groups:',len(china_groups),'largest:',sorted([len(groups[g]) for g in china_groups],reverse=True)[:10],flush=True)
allocation = {g:'train' for g in forced}
counts = Counter()
targets = {'val':0.15*china_total,'calibration':0.15*china_total}
for g in china_groups:
    split = max(targets,key=lambda s:(targets[s]-counts[s])/targets[s])
    if counts[split] < targets[split]:
        allocation[g] = split
        counts[split] += len(groups[g])
    else: allocation[g] = 'train'
manifest_rows = []
for i,r in enumerate(v3_records):
    split = allocation[find(i)]
    row = dict(split=split, country=r['country'], stem=r['stem'], group=find(i), image=str(r['image'].relative_to(V3_SOURCE)), label=str(r['label'].relative_to(V3_SOURCE)), sha256=r['exact'], phash=f"{r['phash']:016x}", boxes=r['boxes'])
    row.update({c:int(r['present'][j]) for j,c in enumerate(V3_CLASSES)})
    manifest_rows.append(row)
v3_df = pd.DataFrame(manifest_rows)
assert (v3_df.groupby('group')['split'].nunique() == 1).all()
assert all(allocation[find(a)]==allocation[find(b)] or 'quarantine' in [allocation[find(a)],allocation[find(b)]] for a,b,d in near_pairs)
v3_ready = {'train','val','calibration'} <= set(v3_df['split']) and all((v3_df.split==s).sum()>=100 for s in ['val','calibration'])
summary = v3_df.groupby(['split','country']).agg(images=('stem','size'),boxes=('boxes','sum'),**{c:(c,'sum') for c in V3_CLASSES})
summary.to_csv(V3_OUT/'class_distribution.csv')
v3_df.to_csv(V3_OUT/'candidate_manifest.csv',index=False)
print(summary.to_string(),flush=True)
if v3_ready:
    assert (v3_df.groupby('split')[V3_CLASSES].sum() > 0).all().all(), 'A class is absent; redesign grouped split'
    v3_df.to_csv(V3_OUT/'manifest.csv',index=False)
    for split in ['train','val','calibration','quarantine']:
        part = v3_df[v3_df.split==split]
        part.to_csv(V3_OUT/(split+'_manifest.csv'),index=False)
        (V3_OUT/(split+'.txt')).write_text('\n'.join(str(V3_SOURCE/p) for p in part.image)+'\n')
    config = dict(path=str(V3_SOURCE),train=str(V3_OUT/'train.txt'),val=str(V3_OUT/'val.txt'),names=dict(enumerate(V3_CLASSES)))
    (V3_OUT/'data_v3.yaml').write_text(yaml.safe_dump(config,sort_keys=False))
    summary = v3_df.groupby(['split','country']).agg(images=('stem','size'),boxes=('boxes','sum'),**{c:(c,'sum') for c in V3_CLASSES})
    summary.to_csv(V3_OUT/'class_distribution.csv')
    
else:
    print('BLOCKED: The conservative similarity audit removed a held-out split. No training-ready YAML or split lists are exported. Review scene/route metadata or visually verify perceptual matches first.',flush=True)
pair_rows = [dict(image_a=v3_records[a]['stem'],image_b=v3_records[b]['stem'],hamming=d,group=find(a),split_a=allocation[find(a)],split_b=allocation[find(b)]) for a,b,d in near_pairs]
pd.DataFrame(pair_rows,columns=['image_a','image_b','hamming','group','split_a','split_b']).to_csv(V3_OUT/'near_duplicate_pairs.csv',index=False)
# Reproducible examples nearest the pHash boundary, for visual inspection.
examples = sorted(near_pairs,key=lambda t:(v3_records[t[0]]['country'] != 'China_MotorBike',-t[2],v3_records[t[0]]['stem']))[:12]
sheet = Image.new('RGB',(640,max(1,len(examples))*160),'white')
draw = ImageDraw.Draw(sheet)
for k,(a,b,d) in enumerate(examples):
    for col,idx in enumerate([a,b]):
        with Image.open(v3_records[idx]['image']) as im:
            tile = ImageOps.contain(im.convert('RGB'),(310,130))
        sheet.paste(tile,(col*320,k*160))
        draw.text((col*320,k*160+132),v3_records[idx]['stem'][:40],fill='black')
    draw.text((0,k*160+146),f'Hamming={d}; assigned={allocation[find(a)]}',fill='black')
sheet.save(V3_OUT/'similarity_review.jpg')
report = dict(seed=V3_SEED,source=str(V3_SOURCE),images=len(v3_df),china_images=china_total,phash_radius=V3_RADIUS,grouping='exact SHA256 plus perceptual connected components; no arbitrary filename blocks',near_pairs=len(near_pairs),quarantined_images=int((v3_df.split=='quarantine').sum()),status='PROVISIONAL_REQUIRES_SCENE_REVIEW' if v3_ready else 'BLOCKED_REQUIRES_SCENE_REVIEW',training_ready=False,test_images_opened=False,seconds=round(time.monotonic()-v3_start,1))
(V3_OUT/'audit.json').write_text(json.dumps(report,indent=2))
(V3_OUT/'README.txt').write_text('CPU audit only. candidate_manifest.csv is diagnostic, not an approved training split. Review perceptual group integrity before approving the candidate split. Inspect similarity_review.jpg and near_duplicate_pairs.csv, then obtain route/scene metadata or refine verified similarity rules. Do not train from the candidate manifest. Previous v2 is preserved in version history. US/local were not opened for this audit.')
print(json.dumps(report,indent=2))
print('Saved diagnostic artifacts:', sorted(p.name for p in V3_OUT.iterdir()))
from IPython.display import display
display(sheet)


## 5. Train YOLO11s on the expanded split and calibrate separately

Authorized exploratory run on the current visual-group split: 22,423 train, 298 validation, 297 calibration. Perceptual grouping does not establish route independence. Start fresh from COCO YOLO11s (640), retaining the v2 optimizer/augmentation recipe, 40 epochs and patience 8. Allow up to 4 hours of training (checked at epoch boundaries); the previous 90-minute cap would reduce the number of epochs on this larger dataset.

Checkpoint selection uses validation detection fitness. Select image-level class thresholds only on calibration, then apply those fixed thresholds to validation. Validation has already selected the checkpoint; both are development results. US/local test images are not evaluated in this run. The old ResNet/YOLO numbers used different splits and cannot be treated as controlled comparisons.

Save best/last weights, history, versions, manifests, calibration thresholds and per-class detection/image metrics. Earlier v2 cells are disabled for Run All; their code remains here and their completed results remain in Version 1. User authorized proceeding with this provisional split; route-review limitations remain recorded.


In [ ]:
import sys, subprocess, importlib.metadata, platform, shutil
from sklearn.metrics import precision_recall_fscore_support
from concurrent.futures import ThreadPoolExecutor
assert v3_ready, 'Split checks did not pass'
assert v3_df.groupby('split').size().to_dict() == {'calibration':297,'train':22423,'val':298}
try:
    assert importlib.metadata.version('ultralytics') == '8.4.177'
except (importlib.metadata.PackageNotFoundError, AssertionError):
    subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics==8.4.177'])
import torch, ultralytics
from ultralytics import YOLO
assert torch.cuda.is_available(), 'GPU required for this training run'
V3_ART = Path('/kaggle/working/artifacts_v3')
V3_ART.mkdir(exist_ok=True)
V3_STAGE = Path('/kaggle/working/expanded_yolo_v3')
for split in ['train','val','calibration']:
    for folder in ['images','labels']:
        (V3_STAGE/split/folder).mkdir(parents=True,exist_ok=True)
def stage_row(row):
    for kind,folder in [('image','images'),('label','labels')]:
        src = V3_SOURCE/row[kind]
        dst = V3_STAGE/row['split']/folder/src.name
        if not dst.exists(): shutil.copy2(src,dst)
with ThreadPoolExecutor(max_workers=4) as pool:
    list(pool.map(stage_row,v3_df.to_dict('records')))
V3_DATA = V3_ART/'data.yaml'
V3_DATA.write_text(yaml.safe_dump(dict(path=str(V3_STAGE),train='train/images',val='val/images',names=dict(enumerate(V3_CLASSES))),sort_keys=False))
shutil.copy2(V3_OUT/'manifest.csv',V3_ART/'manifest.csv')
(V3_ART/'split_sha256.txt').write_text(hashlib.sha256((V3_OUT/'manifest.csv').read_bytes()).hexdigest())
versions = dict(python=platform.python_version(),torch=torch.__version__,ultralytics=ultralytics.__version__,gpu=torch.cuda.get_device_name(0))
(V3_ART/'environment.json').write_text(json.dumps(versions,indent=2))
print('V3_TRAIN_START',versions,flush=True)
v3_model = YOLO('yolo11s.pt')
v3_clock = {}
def v3_start_clock(trainer): v3_clock['start']=time.monotonic()
def v3_budget_stop(trainer):
    if time.monotonic()-v3_clock['start'] >= 4*3600:
        trainer.stop=True
        print('Four-hour training budget reached; saving at epoch boundary.',flush=True)
v3_model.add_callback('on_train_start',v3_start_clock)
v3_model.add_callback('on_fit_epoch_end',v3_budget_stop)
v3_model.train(data=str(V3_DATA),imgsz=640,epochs=40,patience=8,batch=16,
    device=0,workers=2,seed=42,deterministic=True,
    optimizer='AdamW',lr0=0.001,lrf=0.01,cos_lr=True,warmup_epochs=3,weight_decay=0.0005,
    mosaic=1.0,close_mosaic=10,mixup=0.0,scale=0.5,translate=0.1,degrees=0.0,
    fliplr=0.5,flipud=0.0,hsv_h=0.015,hsv_s=0.7,hsv_v=0.4,
    cache=False,amp=True,save=True,save_period=5,plots=True,
    project='/kaggle/working/runs_v3',name='yolo11s_640_expanded_seed42',exist_ok=False)
v3_run=Path(v3_model.trainer.save_dir)
for name in ['best.pt','last.pt']:
    src=v3_run/'weights'/name
    if src.exists(): shutil.copy2(src,V3_ART/('yolo_v3_'+name))
for name in ['results.csv','args.yaml','results.png']:
    if (v3_run/name).exists(): shutil.copy2(v3_run/name,V3_ART/name)
print('V3_TRAIN_COMPLETE',str(v3_run),flush=True)
# Separate calibration and development evaluation; never use US/local here.
v3_best=YOLO(str(V3_ART/'yolo_v3_best.pt'))
det=v3_best.val(data=str(V3_DATA),split='val',imgsz=640,batch=16,device=0,workers=2,plots=True,project='/kaggle/working/evaluation_v3',name='validation')
(V3_ART/'validation_detection.json').write_text(json.dumps({str(k):float(v) for k,v in det.results_dict.items()},indent=2))
def image_scores(split):
    part=v3_df[v3_df.split==split].sort_values('stem')
    stems=part.stem.to_list()
    lookup={s:i for i,s in enumerate(stems)}
    truth=part[V3_CLASSES].to_numpy(dtype=int)
    scores=np.zeros((len(part),4),dtype=float)
    for result in v3_best.predict(source=str(V3_STAGE/split/'images'),imgsz=640,conf=0.001,iou=0.7,max_det=300,device=0,stream=True,verbose=False):
        i=lookup[Path(result.path).stem]
        for c,p in zip(result.boxes.cls.cpu().numpy().astype(int),result.boxes.conf.cpu().numpy()):
            scores[i,c]=max(scores[i,c],float(p))
    np.savez_compressed(V3_ART/(split+'_image_scores.npz'),stems=np.array(stems),truth=truth,scores=scores)
    return truth,scores
cal_y,cal_s=image_scores('calibration')
grid=np.arange(0.05,0.96,0.05)
thresholds=[]
for c in range(4):
    fs=[precision_recall_fscore_support(cal_y[:,c],cal_s[:,c]>=t,average='binary',zero_division=0)[2] for t in grid]
    thresholds.append(float(grid[int(np.argmax(fs))]))
(V3_ART/'inference_config.json').write_text(json.dumps(dict(classes=V3_CLASSES,thresholds=thresholds,threshold_source='calibration only',imgsz=640,conf_floor=0.001,iou=0.7,max_det=300,scene_independence='not verified'),indent=2))
rows=[]
for domain,(truth,scores) in [('calibration',(cal_y,cal_s)),('validation',image_scores('val'))]:
    for mode,ts in [('fixed_0.5',np.full(4,0.5)),('calibration_thresholds',np.array(thresholds))]:
        p,r,f,support=precision_recall_fscore_support(truth,scores>=ts,zero_division=0)
        print(domain,mode,'Macro-F1:',float(f.mean()),flush=True)
        for c,name in enumerate(V3_CLASSES):
            rows.append(dict(domain=domain,mode=mode,cls=name,threshold=float(ts[c]),precision=float(p[c]),recall=float(r[c]),f1=float(f[c]),support=int(support[c])))
pd.DataFrame(rows).to_csv(V3_ART/'image_metrics.csv',index=False)
summary=dict(status='complete',train_images=22423,validation_images=298,calibration_images=297,epochs_completed=len(pd.read_csv(v3_run/'results.csv')),test_evaluated=False,split_limitation='perceptual groups; route independence unverified',detection={str(k):float(v) for k,v in det.results_dict.items()})
(V3_ART/'summary.json').write_text(json.dumps(summary,indent=2))
print('V3_ALL_COMPLETE',json.dumps(summary),flush=True)


## 6. Evaluate generalization on US images

We evaluate the saved v3 model on 4,805 US images excluded from training. We reuse the calibration thresholds without changes and report bounding-box mAP and image-level Macro-F1 separately. No training or threshold tuning is performed on US data.

In [1]:
from pathlib import Path
import sys, subprocess, json, shutil
from concurrent.futures import ThreadPoolExecutor

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "ultralytics==8.4.177"
])

import numpy as np
import pandas as pd
import torch
import yaml
from ultralytics import YOLO
from sklearn.metrics import precision_recall_fscore_support

assert torch.cuda.is_available(), "فعّلي GPU أولًا"

# Locate the saved v3 model.
candidates = [
    Path("/kaggle/input/notebooks/rimasabunasser/"
         "05-roadscan-yolo-v2/artifacts_v3"),
    Path("/kaggle/input/05-roadscan-yolo-v2/artifacts_v3"),
]
matches = [
    p for p in candidates
    if (p / "yolo_v3_best.pt").exists()
]
assert len(matches) == 1, (
    "لم أجد الأوزان في المسار المتوقع. "
    "انسخي مسار yolo_v3_best.pt من قائمة Input."
)
art = matches[0]

config = json.loads((art / "inference_config.json").read_text())
classes = config["classes"]
thresholds = np.array(config["thresholds"])
assert classes == ["D00", "D10", "D20", "D40"]

source = Path(
    "/kaggle/input/datasets/sarvii7/roadscan-yolo/"
    "RoadScan_yolo/test_US"
)
images = sorted((source / "images").glob("*.jpg"))
assert len(images) == 4805, f"عدد صور أمريكا: {len(images)}"

# Stage only US images and labels.
root = Path("/kaggle/working/us_test_v3")
out = Path("/kaggle/working/us_results_v3")
out.mkdir(parents=True, exist_ok=True)
for folder in ["images", "labels"]:
    (root / folder).mkdir(parents=True, exist_ok=True)

def copy_pair(image):
    label = source / "labels" / f"{image.stem}.txt"
    assert label.exists(), f"Missing label: {label.name}"
    for src, folder in [(image, "images"), (label, "labels")]:
        dst = root / folder / src.name
        if not dst.exists():
            shutil.copy2(src, dst)

with ThreadPoolExecutor(max_workers=4) as pool:
    list(pool.map(copy_pair, images))

data = out / "us_data.yaml"
data.write_text(yaml.safe_dump({
    "path": str(root),
    "train": "images",
    "val": "images",
    "test": "images",
    "names": dict(enumerate(classes)),
}, sort_keys=False))

model = YOLO(str(art / "yolo_v3_best.pt"))
imgsz = config["imgsz"]

# Bounding-box detection metrics.
detection = model.val(
    data=str(data), split="test",
    imgsz=imgsz, batch=16, device=0, workers=2,
    plots=True, project=str(out), name="detection"
)
det_metrics = {
    str(k): float(v)
    for k, v in detection.results_dict.items()
}
(out / "us_detection.json").write_text(
    json.dumps(det_metrics, indent=2)
)

# Image-level classification metrics.
stems = [p.stem for p in images]
lookup = {stem: i for i, stem in enumerate(stems)}
truth = np.zeros((len(stems), 4), dtype=int)
scores = np.zeros((len(stems), 4), dtype=float)

for i, stem in enumerate(stems):
    label = root / "labels" / f"{stem}.txt"
    for line in label.read_text().splitlines():
        if line.strip():
            truth[i, int(line.split()[0])] = 1

for result in model.predict(
    source=str(root / "images"),
    imgsz=imgsz,
    conf=config["conf_floor"],
    iou=config["iou"],
    max_det=config["max_det"],
    device=0, stream=True, verbose=False
):
    i = lookup[Path(result.path).stem]
    for cls, conf in zip(
        result.boxes.cls.cpu().numpy().astype(int),
        result.boxes.conf.cpu().numpy()
    ):
        scores[i, cls] = max(scores[i, cls], float(conf))

rows = []
macro_f1 = {}

for mode, limits in [
    ("fixed_0.5", np.full(4, 0.5)),
    ("saved_calibration_thresholds", thresholds),
]:
    p, r, f, support = precision_recall_fscore_support(
        truth, scores >= limits, zero_division=0
    )
    macro_f1[mode] = float(f.mean())
    print(f"\nUS {mode}: Macro-F1 = {f.mean():.4%}")

    for c, name in enumerate(classes):
        rows.append({
            "class": name,
            "mode": mode,
            "threshold": float(limits[c]),
            "precision": float(p[c]),
            "recall": float(r[c]),
            "f1": float(f[c]),
            "support": int(support[c]),
        })

metrics = pd.DataFrame(rows)
metrics.to_csv(out / "us_image_metrics.csv", index=False)

np.savez_compressed(
    out / "us_image_scores.npz",
    stems=np.array(stems), truth=truth, scores=scores
)
(out / "us_summary.json").write_text(json.dumps({
    "images": len(images),
    "macro_f1": macro_f1,
    "detection": det_metrics,
    "thresholds": thresholds.tolist(),
    "thresholds_adjusted_on_US": False,
}, indent=2))

print("\nDetection:", det_metrics)
display(metrics)
print("\nSaved results:", out)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 4.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11s summary (fused): 100 layers, 9,414,348 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1822.1±603.2 MB/s, size: 77.2 KB)
val: Scanning /kaggle/working/us_test_v3/labels... 4805 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4805/4805 1.3Kit/s 3.7s0.0s
val: New cache created: /kaggle/working/us_test_v3/labels.cache
                 Cla

,class,mode,threshold,precision,recall,f1,support
0,D00,fixed_0.5,0.50,0.986817,0.402355,0.571636,3907
1,D10,fixed_0.5,0.50,0.986979,0.161827,0.278063,2342
2,D20,fixed_0.5,0.50,0.796339,0.470907,0.591837,739
3,D40,fixed_0.5,0.50,0.612903,0.163793,0.258503,116
4,D00,saved_calibration_thresholds,0.20,0.941937,0.876120,0.907837,3907
5,D10,saved_calibration_thresholds,0.25,0.913233,0.624680,0.741886,2342
6,D20,saved_calibration_thresholds,0.05,0.396034,0.945873,0.558307,739
7,D40,saved_calibration_thresholds,0.40,0.565217,0.224138,0.320988,116



Saved results: /kaggle/working/us_results_v3


### US results — interpretation

| Metric | China validation | US test |
|---|---:|---:|
| Image-level Macro-F1 (calibrated thresholds) | 92.45% | 63.23% |
| Detection mAP@50 | 84.11% | 47.72% |
| Detection mAP@50–95 | 52.54% | 26.10% |

The US results show a generalization gap. Potholes (D40: F1 32.10%) and alligator cracks (D20: F1 55.83%) need the most attention. The drop alone does not prove overfitting; different image conditions, class balance, and threshold transfer may contribute. China road/scene independence remains unverified. Since US results have now been inspected, future improvements need a fresh, untouched final test set.

## 7. Audit data coverage before the next experiment

We count images, missing labels, and class support by country in the uploaded subset. This identifies gaps before adding more RDD2022 data or changing training. The audit does not retrain the model or change any split. US results are now exploratory; keep a fresh final test set untouched.

In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd

SOURCE = Path('/kaggle/input/datasets/sarvii7/roadscan-yolo/RoadScan_yolo')
assert SOURCE.exists(), f'Missing input: {SOURCE}'
OUT = Path('/kaggle/working/data_coverage_v4')
OUT.mkdir(exist_ok=True)
CLASSES = ['D00', 'D10', 'D20', 'D40']

def country_of(stem):
    for prefix in ['China_MotorBike', 'China_Drone', 'United_States', 'Japan', 'India', 'Czech', 'Norway', 'US']:
        if stem.startswith(prefix):
            return prefix
    return 'UNKNOWN'

counts = {}
for split in ['train', 'val', 'test_US']:
    images = sorted(p for p in (SOURCE / split / 'images').iterdir()
                    if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
    for image in images:
        key = (split, country_of(image.stem))
        c = counts.setdefault(key, Counter())
        c['images'] += 1
        label = SOURCE / split / 'labels' / (image.stem + '.txt')
        if not label.exists():
            c['missing_labels'] += 1
            continue
        present = set()
        for line in label.read_text().splitlines():
            if not line.strip():
                continue
            try:
                fields = line.split()
                cls = int(fields[0])
                assert len(fields) == 5 and 0 <= cls < 4
                coords = list(map(float, fields[1:]))
                assert all(0 <= x <= 1 for x in coords) and coords[2] > 0 and coords[3] > 0
            except (ValueError, AssertionError, IndexError):
                c['invalid_rows'] += 1
                continue
            c[CLASSES[cls] + '_boxes'] += 1
            present.add(cls)
        for cls in present:
            c[CLASSES[cls] + '_positive_images'] += 1
        if not present:
            c['empty_or_invalid_labels'] += 1

columns = ['images', 'missing_labels', 'invalid_rows', 'empty_or_invalid_labels'] + [f'{c}_{suffix}' for c in CLASSES for suffix in ['positive_images', 'boxes']]
coverage = pd.DataFrame([{'source_split': split, 'country': country, **{col: c[col] for col in columns}} for (split, country), c in counts.items()])
coverage.to_csv(OUT / 'coverage_by_country.csv', index=False)
display(coverage)
observed = set(coverage['country'])
print('Country prefixes present:', sorted(observed))
print('Norway present:', 'Norway' in observed)
print('China_Drone present:', 'China_Drone' in observed)
print('Saved:', OUT / 'coverage_by_country.csv')
